# PJ-03: Training Deep Encoders & LLMs on Google Colab

This notebook executes the GPU-intensive training phases of the **PJ-03 Hate Speech Detection & Explainable AI** project:
1. **Phase 1: Deep Multilingual Encoders** — Fine-tuning `google/muril-base-cased` (for English & Hinglish) and `xlm-roberta-base`.
2. **Phase 2: LoRA SFT** — Fine-tuning `Qwen/Qwen2.5-1.5B-Instruct` to output `Label:` + `Rationale:`.
3. **Phase 3: GRPO Reinforcement Learning** — Optimising with the composite reward: $R = \alpha R_{\text{label}} + \beta R_{\text{faith}} - \gamma R_{\text{bias}}$.

> **Colab Runtime Requirement:** Select **Runtime > Change runtime type > T4 GPU** before starting.

## Step 1: Verify GPU & Install Dependencies

In [ ]:
!nvidia-smi

In [ ]:
# Clone repository or pull latest commits safely into /content/pj03
import os
from pathlib import Path

if not Path('/content/pj03').is_dir():
    !git clone https://github.com/devantaris/vivek.git /content/pj03

%cd /content/pj03
!git pull origin main
!pip install --upgrade pip
!pip install -r requirements.txt
!pip install -e .[dev]
!pip install bitsandbytes accelerate peft trl

## Step 2: Ingest Datasets & Prepare Splits

In [ ]:
%cd /content/pj03
# Download public HateXplain and HateCheck datasets
!python scripts/download_data.py

# Build processed splits in unified schema (toxic_abusive mode: 1=hate+offensive, 0=normal)
!python scripts/build_data.py --mode toxic_abusive

# Build Phase-2 SFT prompt-completion pairs
!python scripts/build_sft.py

## Step 3: Phase 1 — Train MuRIL (Multilingual & Hinglish Contextual Encoder)
`google/muril-base-cased` was trained on 17 Indian languages and Romanized Hinglish. This provides immediate, dramatic improvements over the word-count baseline.

In [ ]:
%cd /content/pj03
!python scripts/run_baseline.py --model encoder \
    baselines.encoders.smoke.enabled=false \
    baselines.encoders.models='[google/muril-base-cased]' \
    baselines.encoders.batch_size=16 \
    baselines.encoders.epochs=3

## Step 4: Phase 2 — LoRA Supervised Fine-Tuning (SFT)
Fine-tunes `Qwen/Qwen2.5-1.5B-Instruct` on the formatted rationale dataset. It trains with completion-only loss masking so the model learns to emit:
```text
Label: <0|1>
Rationale: <the key abusive words or reasoning>
```

In [ ]:
%cd /content/pj03
!python scripts/run_sft.py \
    sft.model_name='Qwen/Qwen2.5-1.5B-Instruct' \
    sft.smoke.enabled=false \
    sft.batch_size=8 \
    sft.gradient_accumulation=2 \
    sft.epochs=3

## Step 5: Phase 3 — GRPO Post-Training with Composite Reward
Uses Hugging Face TRL GRPOTrainer to align the SFT checkpoint with our composite reward:
$$R = \alpha R_{\text{label}} + \beta R_{\text{faith}} - \gamma R_{\text{bias}}$$
- Penalises spurious identity-term mentions ($R_{\text{bias}}$)
- Rewards faithful explanations ($R_{\text{faith}}$)

In [ ]:
%cd /content/pj03
!python scripts/run_reward.py \
    train.model_name='checkpoints/sft' \
    reward.weights.alpha=1.0 \
    reward.weights.beta=0.5 \
    reward.weights.gamma=0.3 \
    train.epochs=1 \
    train.batch_size=4 \
    train.num_generations=8

## Step 6: Review Benchmarks & Download Checkpoints

In [ ]:
%cd /content/pj03
# View aggregate benchmark results table
!python scripts/summarize_results.py

In [ ]:
%cd /content/pj03
# Zip results and checkpoints to download to your local machine
!zip -r pj03_trained_models.zip data/processed/results/ checkpoints/
from google.colab import files
files.download('pj03_trained_models.zip')